# ENCODE TF-binding reference — end-to-end

1. **search** – query ENCODE, save raw JSON + query metadata  
2. **extract** – filter → `manifest.tsv`, download BEDs (md5-checked, parallel)  
3. **stats** – per-TF / per-biosample tables  
4. **merge (R)** – per-TF `reduce()` → `GRangesList` `.rds`, with experiment-support counts  
5. **inspect** – look at the merge stats

Steps 1–3 call functions from `encode_tf_ref.py`; step 4 calls `tf_merge_to_grl.R` via `Rscript`.
Keep both scripts next to this notebook (or edit `SCRIPT_DIR`).

In [ ]:
import sys, os, json
import pandas as pd

SCRIPT_DIR = "."                      # where encode_tf_ref.py / tf_merge_to_grl.R live
sys.path.insert(0, SCRIPT_DIR)
import encode_tf_ref as etr

ASSEMBLY    = "mm10"                  # or "GRCh38"
RFA         = "ENCODE4"               # None to include ENCODE3 etc.
OUTDIR      = f"ref_{ASSEMBLY}_{RFA or 'all'}"
OUTPUT_TYPE = "conservative IDR thresholded peaks"
DEDUP       = "preferred"             # preferred | latest | none
JOBS        = 8
MIN_SUPPORT = 1                       # R merge: keep regions seen in >= k experiments
OUTPREFIX   = os.path.join(OUTDIR, f"{ASSEMBLY}_TF")

## 1. Search

In [ ]:
search_json = etr.step_search(ASSEMBLY, OUTDIR, RFA, [OUTPUT_TYPE])
json.load(open(os.path.join(OUTDIR, "search_meta.json")))

## 2. Extract + download
Re-running is safe: files already on disk (with matching md5) are skipped.

In [ ]:
manifest = etr.step_extract(search_json, OUTDIR, formats=["bed"], output_type=OUTPUT_TYPE,
                            dedup=DEDUP, download=True, jobs=JOBS)
man = pd.read_csv(manifest, sep="\t")
print(man["download_status"].value_counts())
man.head()

## 3. Stats
`compare_output_types=True` also queries ENCODE for how many targets have conservative vs optimal IDR peaks (what `stats_tf_based.py` did).

In [ ]:
summary = etr.step_stats(manifest, OUTDIR, compare_output_types=True, assembly=ASSEMBLY, rfa=RFA)
per_tf = pd.read_csv(os.path.join(OUTDIR, "stats_per_target.tsv"), sep="\t")
per_tf.sort_values("n_experiments", ascending=False).head(20)

In [ ]:
per_bio = pd.read_csv(os.path.join(OUTDIR, "stats_per_biosample.tsv"), sep="\t")
ax = per_bio.head(25).plot.barh(x="biosample", y="n_targets", figsize=(6, 7), legend=False)
ax.invert_yaxis(); ax.set_xlabel("# targets");

## 4. Merge per TF (R)
Requires R with `data.table`, `GenomicRanges`, and `R.utils` (for reading `.bed.gz`).  
Alternative: with `rpy2` installed you can `%load_ext rpy2.ipython` and run the R code in a `%%R` cell instead.

In [ ]:
cmd = (f"Rscript {SCRIPT_DIR}/tf_merge_to_grl.R --manifest {manifest} --outprefix {OUTPREFIX} "
       f"--min-support {MIN_SUPPORT} --std-chrom TRUE --write-bed FALSE --genome {ASSEMBLY}")
print(cmd)
!{cmd}

## 5. Inspect merge stats

In [ ]:
ms = pd.read_csv(f"{OUTPREFIX}.merge_stats.tsv", sep="\t")
ms["compression"] = ms["n_peaks_raw"] / ms["n_regions_merged"]
ms.sort_values("n_regions_kept", ascending=False).head(20)

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].scatter(ms["n_experiments"], ms["n_regions_kept"], s=8)
axs[0].set(xscale="log", yscale="log", xlabel="# experiments", ylabel="# merged regions")
axs[1].hist(ms["median_width_kept"].dropna(), bins=50)
axs[1].set(xlabel="median merged region width (bp)", ylabel="# TFs")
plt.tight_layout()